# Microsoft Agent Framework를 이용한 휴먼 인더 루프 워크플로우

## 🎯 학습 목표

이 노트북에서는 Microsoft Agent Framework의 `RequestInfoExecutor`를 사용하여 **휴먼 인더 루프(human-in-the-loop)** 워크플로우를 구현하는 방법을 배웁니다. 이 강력한 패턴은 AI 워크플로우를 일시 중지하여 인간의 입력을 수집할 수 있게 하여, 에이전트를 인터랙티브하게 만들고 중요한 결정을 인간이 제어할 수 있도록 합니다.

## 🔄 휴먼 인더 루프란 무엇인가?

<strong>휴먼 인더 루프(HITL)</strong>는 AI 에이전트가 실행을 일시 중지하고 계속하기 전에 인간으로부터 입력을 요청하는 설계 패턴입니다. 이는 다음과 같은 상황에 필수적입니다:

- ✅ **중요한 결정** - 중요한 행동을 취하기 전에 인간의 승인을 받음
- ✅ **모호한 상황** - AI가 불확실할 때 인간이 명확히 함
- ✅ **사용자 선호** - 여러 옵션 중 사용자에게 선택을 요청함
- ✅ **규정 준수 및 안전** - 규제된 작업에 대해 인간의 감독을 보장함
- ✅ **인터랙티브 경험** - 사용자 입력에 반응하는 대화형 에이전트 구축

## 🏗️ Microsoft Agent Framework에서의 동작 방식

프레임워크는 HITL을 위한 세 가지 주요 구성 요소를 제공합니다:

1. **`RequestInfoExecutor`** - 워크플로우를 일시 중지하고 `RequestInfoEvent`를 발생시키는 특수 실행기
2. **`RequestInfoMessage`** - 인간에게 보내는 타입화된 요청 페이로드의 기본 클래스
3. **`RequestResponse`** - `request_id`를 이용해 인간의 응답을 원래 요청과 연관시킴

**워크플로우 패턴:**
```
Agent detects need for input
    ↓
Sends message to RequestInfoExecutor
    ↓
Workflow pauses & emits RequestInfoEvent
    ↓
Application collects human input (console, UI, etc.)
    ↓
Application sends RequestResponse via send_responses_streaming()
    ↓
Workflow resumes with human input
```

## 🏨 예제: 사용자 확인을 포함한 호텔 예약

대체 목적지를 제안하기 <strong>전에</strong> 인간 확인을 추가하여 조건부 워크플로우를 확장할 것입니다:

1. 사용자가 목적지(예: "파리")를 요청함
2. `availability_agent`가 객실 이용 가능 여부를 확인함
3. <strong>객실 없음</strong>인 경우 → `confirmation_agent`가 "대체 목적지를 보시겠습니까?"라고 묻기
4. `RequestInfoExecutor`를 사용하여 워크플로우를 **일시 중지**
5. 사용자 콘솔 입력을 통해 **인간이 "예" 또는 "아니오"로 응답**
6. `decision_manager`가 응답에 따라 경로 지정:
   - <strong>예</strong> → 대체 목적지 표시
   - <strong>아니오</strong> → 예약 요청 취소
7. 최종 결과 표시

이는 사용자가 에이전트의 제안에 대해 제어할 수 있게 하는 방법을 보여줍니다!

---

시작해 봅시다! 🚀


## 1단계: 필요한 라이브러리 가져오기

표준 에이전트 프레임워크 구성 요소들과 <strong>휴먼 인 더 루프 전용 클래스</strong>를 가져옵니다:
- `RequestInfoExecutor` - 휴먼 입력을 위해 워크플로우를 일시정지하는 실행기
- `RequestInfoEvent` - 휴먼 입력 요청 시 발생하는 이벤트
- `RequestInfoMessage` - 유형화된 요청 페이로드의 기본 클래스
- `RequestResponse` - 휴먼 응답과 요청을 연결함
- `WorkflowOutputEvent` - 워크플로우 출력을 감지하는 이벤트


In [ ]:
import asyncio
import json
import os
from dataclasses import dataclass
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    ChatMessage,
    Executor,
    RequestInfoEvent,          # NEW: Event when human input is requested
    RequestInfoExecutor,       # NEW: Executor that gathers human input
    RequestInfoMessage,        # NEW: Base class for request payloads
    RequestResponse,           # NEW: Correlates response with request
    Role,
    WorkflowBuilder,
    WorkflowContext,
    WorkflowOutputEvent,       # NEW: Event for workflow outputs
    WorkflowRunState,          # NEW: Enum of workflow run states
    WorkflowStatusEvent,       # NEW: Event for run state changes
    ai_function,
    executor,
    handler,                   # NEW: Decorator for executor methods
)

# 🤖 Azure OpenAI client integration
from agent_framework.openai import OpenAIChatClient
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")
print("🔄 Human-in-the-loop components loaded: RequestInfoExecutor, RequestInfoEvent, RequestResponse")

✅ All imports successful!
🔄 Human-in-the-loop components loaded: RequestInfoExecutor, RequestInfoEvent, RequestResponse


## 2단계: 구조화된 출력용 Pydantic 모델 정의

이 모델들은 에이전트가 반환할 <strong>스키마</strong>를 정의합니다. 조건부 워크플로우의 모든 모델을 유지하고 다음을 추가합니다:

**휴먼 인 더 루프용 신규:**
- `HumanFeedbackRequest` - 인간에게 전송하는 요청 페이로드를 정의하는 `RequestInfoMessage`의 서브클래스
  - `prompt`(질문할 내용)와 `destination`(사용할 수 없는 도시와 관련된 컨텍스트)를 포함합니다


In [22]:
# Existing models from conditional workflow
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""
    destination: str
    has_availability: bool
    message: str


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""
    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""
    destination: str
    action: str
    message: str


# NEW: Pydantic model for agent's response format
class ConfirmationQuestion(BaseModel):
    """
    Pydantic model used by confirmation_agent's response_format.
    This is what the agent will output as JSON.
    """
    question: str  # The question to ask the user
    destination: str  # The unavailable destination for context


# NEW: Dataclass for RequestInfoExecutor
@dataclass
class HumanFeedbackRequest(RequestInfoMessage):
    """
    Request sent to RequestInfoExecutor asking if user wants alternatives.
    
    MUST be a dataclass subclassing RequestInfoMessage for type compatibility.
    This is what gets sent to the RequestInfoExecutor.
    """
    prompt: str = ""  # The question to ask the user
    destination: str = ""  # The unavailable destination for context


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")
print("   - ConfirmationQuestion (agent response format) 🆕")
print("   - HumanFeedbackRequest (RequestInfoMessage for HITL) 🆕")

✅ Pydantic models defined:
   - BookingCheckResult (availability check)
   - AlternativeResult (alternative suggestion)
   - BookingConfirmation (booking confirmation)
   - ConfirmationQuestion (agent response format) 🆕
   - HumanFeedbackRequest (RequestInfoMessage for HITL) 🆕


## 3단계: 호텔 예약 도구 만들기

조건부 워크플로와 동일한 도구 - 목적지에 객실이 있는지 확인합니다.


In [23]:
@ai_function(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.
    
    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @ai_function decorator")

✅ hotel_booking tool created with @ai_function decorator


## 4단계: 라우팅을 위한 조건 함수 정의

휴먼 인 더 루프 워크플로우를 위해 <strong>네 가지 조건 함수</strong>가 필요합니다:

**기존 조건 워크플로우에서:**
1. `has_availability_condition` - 호텔이 이용 가능할 때 라우팅
2. `no_availability_condition` - 호텔이 이용 불가할 때 라우팅

**휴먼 인 더 루프를 위해 새롭게 추가:**
3. `user_wants_alternatives_condition` - 사용자가 대안에 대해 "예"라고 말할 때 라우팅
4. `user_declines_alternatives_condition` - 사용자가 대안에 대해 "아니오"라고 말할 때 라우팅


In [24]:
# Existing condition functions from conditional workflow
def has_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels ARE available."""
    if not isinstance(message, AgentExecutorResponse):
        return True

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}
            </div>
        """)
        )
        return result.has_availability
    except Exception as e:
        display(HTML(f"""<div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'><strong>⚠️  Error:</strong> {str(e)}</div>"""))
        return False


def no_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels are NOT available."""
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )
        return not result.has_availability
    except Exception as e:
        return False


# NEW: Condition functions for human-in-the-loop routing
def user_wants_alternatives_condition(message: Any) -> bool:
    """
    Condition for routing when user WANTS to see alternatives.
    
    Checks the AgentExecutorRequest sent by decision_manager.
    """
    # Check if it's an AgentExecutorRequest (what decision_manager sends)
    if isinstance(message, AgentExecutorRequest):
        # Check the message text to determine user's choice
        if message.messages and len(message.messages) > 0:
            msg_text = message.messages[0].text.lower()
            wants_alternatives = "wants to see alternative" in msg_text or "want to see alternative" in msg_text
            
            display(
                HTML(f"""
                <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
                    <strong>🔍 User Decision:</strong> User wants alternatives = <strong>{wants_alternatives}</strong>
                </div>
            """)
            )
            
            return wants_alternatives
    
    return False
def user_declines_alternatives_condition(message: Any) -> bool:
    """
    Condition for routing when user DECLINES alternatives.
    
    Checks the AgentExecutorRequest sent by decision_manager.
    """
    # Check if it's an AgentExecutorRequest (what decision_manager sends)
    if isinstance(message, AgentExecutorRequest):
        # Check the message text to determine user's choice
        if message.messages and len(message.messages) > 0:
            msg_text = message.messages[0].text.lower()
            declined = "declined" in msg_text or "has declined" in msg_text
            
            display(
                HTML(f"""
                <div style='padding: 12px; background: #fce4ec; border-left: 4px solid #c2185b; border-radius: 4px; margin: 10px 0;'>
                    <strong>🚫 User Decision:</strong> User declined alternatives = <strong>{declined}</strong>
                </div>
            """)
            )
            
            return declined
    
    return False
print("✅ Condition functions defined:")
print("   - has_availability_condition (routes when rooms exist)")
print("   - no_availability_condition (routes when no rooms)")
print("   - user_wants_alternatives_condition (routes when user says yes) 🆕")
print("   - user_declines_alternatives_condition (routes when user says no) 🆕")

✅ Condition functions defined:
   - has_availability_condition (routes when rooms exist)
   - no_availability_condition (routes when no rooms)
   - user_wants_alternatives_condition (routes when user says yes) 🆕
   - user_declines_alternatives_condition (routes when user says no) 🆕


## 5단계: Decision Manager Executor 만들기

이것이 <strong>human-in-the-loop 패턴의 핵심</strong>입니다! `DecisionManager`는 다음을 수행하는 맞춤형 `Executor`입니다:

1. `RequestResponse` 객체를 통해 **사람의 피드백을 받습니다**
2. 사용자의 결정(예/아니오)을 <strong>처리합니다</strong>
3. 적절한 에이전트에게 메시지를 보내 워크플로를 **경로 지정합니다**

주요 특징:
- `@handler` 데코레이터를 사용하여 메서드를 워크플로 단계로 노출
- 원본 요청과 사용자의 답변을 모두 포함하는 `RequestResponse[HumanFeedbackRequest, str]` 수신
- 조건 함수를 트리거하는 간단한 “예” 또는 “아니오” 메시지를 생성


In [25]:
class DecisionManager(Executor):
    """
    Coordinates workflow routing based on human feedback.
    
    This executor receives RequestResponse objects from the RequestInfoExecutor
    and makes routing decisions by sending simple messages that trigger
    condition functions.
    """

    def __init__(self, id: str | None = None):
        super().__init__(id=id or "decision_manager")

    @handler
    async def on_human_feedback(
        self,
        feedback: RequestResponse[HumanFeedbackRequest, str],
        ctx: WorkflowContext[AgentExecutorRequest],
    ) -> None:
        """
        Process human feedback and let the workflow route based on conditions.
        
        The RequestResponse contains:
        - feedback.data: The user's string reply (e.g., "yes" or "no")
        - feedback.original_request: The HumanFeedbackRequest with context
        
        This handler just displays feedback and passes the RequestResponse through.
        The routing is done by condition functions on the edges.
        """
        user_reply = (feedback.data or "").strip().lower()
        destination = getattr(feedback.original_request, "destination", "unknown")

        display(
            HTML(f"""
            <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
                <strong>🎯 Decision Manager:</strong> Processing user reply: <strong>"{user_reply}"</strong> for {destination}
            </div>
        """)
        )

        if user_reply == "yes":
            display(
                HTML("""
                <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                    <strong>➡️  Routing:</strong> User wants alternatives → Will route to alternative_agent
                </div>
            """)
            )
            # Create and send a message for the alternative_agent
            user_msg = ChatMessage(
                Role.USER,
                text=f"The user wants to see alternative destinations near {destination}. Please suggest one.",
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))
        
        elif user_reply == "no":
            display(
                HTML("""
                <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                    <strong>🚫 Routing:</strong> User declined alternatives → Will route to cancellation_agent
                </div>
            """)
            )
            # Create and send a message for the cancellation_agent
            user_msg = ChatMessage(
                Role.USER,
                text="The user has declined to see alternatives. Please acknowledge their decision.",
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))
        
        else:
            # Handle unexpected input - treat as decline
            display(
                HTML(f"""
                <div style='padding: 12px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                    <strong>⚠️  Warning:</strong> Unexpected input "{user_reply}" - treating as decline
                </div>
            """)
            )
            user_msg = ChatMessage(
                Role.USER,
                text="The user has declined to see alternatives. Please acknowledge their decision.",
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))


print("✅ DecisionManager executor created with @handler method for human feedback")

✅ DecisionManager executor created with @handler method for human feedback


## 6단계: 사용자 정의 표시 실행기 생성

조건부 워크플로우의 동일한 표시 실행기 - 최종 결과를 워크플로우 출력으로 반환합니다.


In [26]:
@executor(id="prepare_human_request")
async def prepare_human_request(
    response: AgentExecutorResponse, 
    ctx: WorkflowContext[HumanFeedbackRequest]
) -> None:
    """
    Transform agent response into HumanFeedbackRequest for RequestInfoExecutor.
    
    This executor bridges the type gap between:
    - confirmation_agent outputs AgentExecutorResponse with ConfirmationQuestion JSON
    - request_info_executor expects HumanFeedbackRequest (RequestInfoMessage dataclass)
    """
    display(
        HTML("""
        <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
            <strong>🔄 Transform:</strong> Converting ConfirmationQuestion to HumanFeedbackRequest
        </div>
    """)
    )
    
    # Parse the agent's Pydantic output (ConfirmationQuestion)
    confirmation = ConfirmationQuestion.model_validate_json(response.agent_run_response.text)
    
    # Convert to HumanFeedbackRequest dataclass for RequestInfoExecutor
    feedback_request = HumanFeedbackRequest(
        prompt=confirmation.question,
        destination=confirmation.destination
    )
    
    # Send the properly typed RequestInfoMessage to the RequestInfoExecutor
    await ctx.send_message(feedback_request)


@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """
    Display the final result as workflow output.
    
    This executor receives the final agent response and yields it as the workflow output.
    """
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ prepare_human_request executor created with @executor decorator")
print("✅ display_result executor created with @executor decorator")

✅ prepare_human_request executor created with @executor decorator
✅ display_result executor created with @executor decorator


## 7단계: 환경 변수 로드

LLM 클라이언트(Microsoft Foundry / Azure OpenAI)를 구성합니다.


In [ ]:
# Load environment variables
load_dotenv()

from azure.identity import AzureCliCredential

# Azure OpenAI via the Responses API. Sign in with `az login` for keyless Entra ID auth.
# GitHub Models is deprecated (retiring July 2026) and does not support the Responses API,
# so this sample calls Azure OpenAI directly. OpenAIChatClient uses the Responses API.
chat_client = OpenAIChatClient(
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    credential=AzureCliCredential(),
    model_id=os.environ["AZURE_OPENAI_DEPLOYMENT"],
)

print("✅ Chat client configured with Azure OpenAI (Responses API)")


✅ Chat client configured with GitHub Models


## 8단계: AI 에이전트 및 실행기 생성

우리는 <strong>여섯 개의 워크플로우 구성요소</strong>를 생성합니다:

**에이전트 (AgentExecutor로 래핑됨):**
1. **availability_agent** - 도구를 사용해 호텔 이용 가능 여부 확인
2. **confirmation_agent** - 🆕 사람 확인 요청 준비
3. **alternative_agent** - 대체 도시 제안 (사용자가 예라고 할 때)
4. **booking_agent** - 예약 권장 (객실 이용 가능 시)
5. **cancellation_agent** - 🆕 취소 메시지 처리 (사용자가 아니라고 할 때)

**특수 실행기:**
6. **request_info_executor** - 🆕 사용자 입력을 위해 워크플로우를 일시 중지하는 `RequestInfoExecutor`
7. **decision_manager** - 🆕 사람 응답에 따라 라우팅하는 맞춤 실행기 (위에서 이미 정의됨)


In [ ]:
# Agent 1: Check availability with tool (same as conditional workflow)
availability_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), and message (string). "
            "The message should summarize the availability status."
        ),
        tools=[hotel_booking],
        response_format=BookingCheckResult,
    ),
    id="availability_agent",
)

# Agent 2: NEW - Prepare human confirmation request
confirmation_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful assistant. The user's requested destination has no available hotel rooms. "
            "Create a polite message asking if they would like to see alternative destinations nearby. "
            "Return a JSON with: destination (the unavailable city), and question (a friendly yes/no question). "
            "Keep the question concise and friendly."
        ),
        response_format=ConfirmationQuestion,  # Use Pydantic model for agent output
    ),
    id="confirmation_agent",
)

# Agent 3: Suggest alternative (when user says yes)
alternative_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        response_format=AlternativeResult,
    ),
    id="alternative_agent",
)

# Agent 4: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        response_format=BookingConfirmation,
    ),
    id="booking_agent",
)

# Agent 5: NEW - Handle cancellation when user declines alternatives
class CancellationMessage(BaseModel):
    """Message when user declines alternatives."""
    status: str
    message: str

cancellation_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful assistant. The user has declined to see alternative hotel destinations. "
            "Create a polite cancellation message. "
            "Return JSON with: status (should be 'cancelled'), and message (a friendly acknowledgment). "
            "Keep the message brief and understanding."
        ),
        response_format=CancellationMessage,
    ),
    id="cancellation_agent",
)

# NEW: RequestInfoExecutor - pauses workflow to gather human input
request_info_executor = RequestInfoExecutor(id="request_info")

# NEW: DecisionManager instance - routes based on human feedback
decision_manager = DecisionManager(id="decision_manager")

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created Workflow Components:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - Checks availability with hotel_booking tool</li>
            <li><strong>confirmation_agent</strong> 🆕 - Prepares human confirmation request</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
            <li><strong>cancellation_agent</strong> 🆕 - Handles user declining alternatives</li>
            <li><strong>request_info_executor</strong> 🆕 - Pauses workflow for human input</li>
            <li><strong>decision_manager</strong> 🆕 - Routes based on human response</li>
        </ul>
    </div>
""")
)


## 9단계: 사람 개입 워크플로우 빌드하기

이제 사람 개입 경로를 포함한 <strong>조건부 라우팅</strong>과 함께 워크플로우 그래프를 구성합니다:

**워크플로우 구조:**
```
availability_agent (START)
        ↓
   Evaluate conditions
        ↙                    ↘
[no_availability]        [has_availability]
        ↓                        ↓
confirmation_agent          booking_agent
        ↓                        ↓
prepare_human_request      display_result
        ↓
request_info_executor (PAUSE)
        ↓
decision_manager
   ↙         ↘
[yes]        [no]
   ↓           ↓
alternative  cancellation
   ↓           ↓
display_result
```

**주요 엣지:**
- `availability_agent → confirmation_agent` (방이 없을 때)
- `confirmation_agent → prepare_human_request` (타입 변환)
- `prepare_human_request → request_info_executor` (사람 대기)
- `request_info_executor → decision_manager` (항상 - RequestResponse 제공)
- `decision_manager → alternative_agent` (사용자가 "예"라고 할 때)
- `decision_manager → cancellation_agent` (사용자가 "아니요"라고 할 때)
- `availability_agent → booking_agent` (방이 있을 때)
- 모든 경로는 `display_result`에서 끝납니다


In [29]:
# Build the workflow with human-in-the-loop routing
workflow = (
    WorkflowBuilder()
    .set_start_executor(availability_agent)
    
    # NO AVAILABILITY PATH (with human-in-the-loop)
    .add_edge(availability_agent, confirmation_agent, condition=no_availability_condition)
    .add_edge(confirmation_agent, prepare_human_request)  # Transform to HumanFeedbackRequest
    .add_edge(prepare_human_request, request_info_executor)  # Send to RequestInfoExecutor
    .add_edge(request_info_executor, decision_manager)    # Always goes to decision manager
    
    # Decision manager routes based on user response
    .add_edge(decision_manager, alternative_agent, condition=user_wants_alternatives_condition)
    .add_edge(decision_manager, cancellation_agent, condition=user_declines_alternatives_condition)
    .add_edge(alternative_agent, display_result)
    .add_edge(cancellation_agent, display_result)
    
    # HAS AVAILABILITY PATH (no human input needed)
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Human-in-the-Loop Routing:</strong><br>
            • If <strong>NO availability</strong> → confirmation_agent → prepare_human_request → request_info_executor → <strong>PAUSE FOR HUMAN</strong> → decision_manager<br>
            &nbsp;&nbsp;• If user says <strong>YES</strong> → alternative_agent → display_result<br>
            &nbsp;&nbsp;• If user says <strong>NO</strong> → cancellation_agent → display_result<br>
            • If <strong>availability</strong> → booking_agent → display_result (no human input needed)
        </p>
    </div>
""")
)

## 단계 10: 테스트 케이스 1 실행 - 가용성 없는 도시 (인간 확인이 필요한 파리)

이 테스트는 <strong>완전한 인간 개입 루프 사이클</strong>을 시연합니다:

1. 파리에서 호텔 요청
2. availability_agent 확인 → 객실 없음
3. confirmation_agent가 사용자 대면 질문 생성
4. request_info_executor가 <strong>작업 흐름을 일시 중지</strong>하고 `RequestInfoEvent` 발생
5. **애플리케이션이 이벤트 감지 후 콘솔에서 사용자에게 프롬프트 표시**
6. 사용자가 "yes" 또는 "no" 입력
7. 애플리케이션이 `send_responses_streaming()`을 통해 응답 전송
8. decision_manager가 응답에 따라 경로 지정
9. 최종 결과 표시

**핵심 패턴:**
- 첫 번째 반복에는 `workflow.run_stream()` 사용
- 이후 반복에는 `workflow.send_responses_streaming(pending_responses)` 사용
- 인간 입력 필요 시점 감지를 위해 `RequestInfoEvent` 수신
- 최종 결과 캡처를 위해 `WorkflowOutputEvent` 수신


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Paris (No Availability - Human-in-the-Loop)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → confirmation_agent → request_info_executor → <strong>PAUSE</strong> → decision_manager → (depends on user input)</p>
    </div>
""")
)

# Create request for Paris
request_paris = AgentExecutorRequest(
    messages=[ChatMessage(Role.USER, text="I want to book a hotel in Paris")], 
    should_respond=True
)

# Human-in-the-loop execution pattern
pending_responses: dict[str, str] | None = None
completed = False
workflow_output: str | None = None

print("\n🔄 Starting human-in-the-loop workflow...")
print("=" * 60)

while not completed:
    # First iteration uses run_stream with the request
    # Subsequent iterations use send_responses_streaming with collected human responses
    if pending_responses:
        print(f"\n📤 Sending human responses: {pending_responses}")
        stream = workflow.send_responses_streaming(pending_responses)
        pending_responses = None  # Clear immediately after sending
    else:
        print(f"\n🚀 Starting workflow with request: 'I want to book a hotel in Paris'")
        stream = workflow.run_stream(request_paris)
    
    # Collect all events from this iteration
    events = [event async for event in stream]
    
    # Process events
    requests: list[tuple[str, str]] = []  # (request_id, prompt)
    
    for event in events:
        # Check for human input requests
        if isinstance(event, RequestInfoEvent) and isinstance(event.data, HumanFeedbackRequest):
            print(f"\n⏸️  WORKFLOW PAUSED - Human input requested!")
            print(f"   Request ID: {event.request_id}")
            print(f"   Destination: {event.data.destination}")
            requests.append((event.request_id, event.data.prompt))
        
        # Check for workflow outputs
        elif isinstance(event, WorkflowOutputEvent):
            workflow_output = str(event.data)
            completed = True
            print(f"\n✅ Workflow completed with output!")
    
    # If we have human requests, prompt the user
    if requests and not completed:
        responses: dict[str, str] = {}
        for req_id, prompt in requests:
            print(f"\n{'='*60}")
            print(f"💬 QUESTION FOR YOU:")
            print(f"   {prompt}")
            print(f"{'='*60}")
            
            # Get user input (in notebook, this will pause execution)
            answer = input("👉 Enter 'yes' or 'no': ").strip().lower()
            
            print(f"\n📝 You answered: {answer}")
            responses[req_id] = answer
        
        pending_responses = responses

print(f"\n{'='*60}")
print(f"🏆 FINAL WORKFLOW OUTPUT:")
print(f"{'='*60}")

# Display final result
if workflow_output:
    # Try to parse as JSON for pretty display
    try:
        result_data = json.loads(workflow_output)
        if "alternative_destination" in result_data:
            result_obj = AlternativeResult.model_validate_json(workflow_output)
            display(
                HTML(f"""
                <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px; box-shadow: 0 4px 12px rgba(255,165,0,0.3); margin: 20px 0;'>
                    <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 WORKFLOW RESULT</h3>
                    <div style='background: white; padding: 20px; border-radius: 8px;'>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>User Decision:</strong> ✅ Accepted alternatives</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Alternative Suggestion:</strong> 🏨 {result_obj.alternative_destination}</p>
                        <p style='margin: 0; font-size: 14px; color: #666;'><strong>Reason:</strong> {result_obj.reason}</p>
                    </div>
                </div>
            """)
            )
        else:
            # User declined
            display(
                HTML(f"""
                <div style='padding: 25px; background: linear-gradient(135deg, #f44336 0%, #e91e63 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(244,67,54,0.3); margin: 20px 0;'>
                    <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT</h3>
                    <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>User Decision:</strong> 🚫 Declined alternatives</p>
                        <p style='margin: 0; font-size: 14px; color: #666;'><strong>Result:</strong> Booking request cancelled</p>
                    </div>
                </div>
            """)
            )
    except:
        print(workflow_output)


🔄 Starting human-in-the-loop workflow...

🚀 Starting workflow with request: 'I want to book a hotel in Paris'



⏸️  WORKFLOW PAUSED - Human input requested!
   Request ID: 032c8fce-b9d1-400e-ba8d-afd2248e2926
   Destination: Paris

💬 QUESTION FOR YOU:
   Unfortunately, there are no rooms available in Paris. Would you like to explore nearby alternative destinations?

📝 You answered: yes

📤 Sending human responses: {'032c8fce-b9d1-400e-ba8d-afd2248e2926': 'yes'}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'

📝 You answered: yes

📤 Sending human responses: {'032c8fce-b9d1-400e-ba8d-afd2248e2926': 'yes'}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'



⏸️  WORKFLOW PAUSED - Human input requested!
   Request ID: cf48dad0-ee5e-4f60-8806-341a7a292bd4
   Destination: Paris

💬 QUESTION FOR YOU:
   I'm sorry to inform you that there are no available hotel rooms in Paris. Would you like me to suggest nearby alternative destinations?

📝 You answered: 

📤 Sending human responses: {'cf48dad0-ee5e-4f60-8806-341a7a292bd4': ''}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'

📝 You answered: 

📤 Sending human responses: {'cf48dad0-ee5e-4f60-8806-341a7a292bd4': ''}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'


## 11단계: 테스트 케이스 2 실행 - 이용 가능 지역이 있는 도시 (스톡홀름 - 인간 입력 불필요)

이 테스트는 방이 있을 때의 <strong>직접 경로</strong>를 보여줍니다:

1. 스톡홀름 호텔 요청
2. availability_agent 확인 → 방 있음 ✅
3. booking_agent가 예약 제안
4. display_result가 확인 표시
5. **인간 입력 불필요!**

방이 있을 경우 워크플로우가 인간 개입 경로를 완전히 우회합니다.


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 2: Stockholm (Has Availability - No Human Input)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → booking_agent → display_result (direct, no pause)</p>
    </div>
""")
)

# Create request for Stockholm
request_stockholm = AgentExecutorRequest(
    messages=[ChatMessage(Role.USER, text="I want to book a hotel in Stockholm")], 
    should_respond=True
)

# Run the workflow (should complete without human input)
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT (Stockholm - No Human Input)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0 0 10px 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
                <p style='margin: 10px 0 0 0; font-size: 12px; color: #999; font-style: italic;'>Note: No human input was requested because rooms were available!</p>
            </div>
        </div>
    """)
    )

## 주요 내용 및 휴먼 인 더 루프 모범 사례

### ✅ 학습 내용:

#### 1. **RequestInfoExecutor 패턴**
마이크로소프트 에이전트 프레임워크의 휴먼 인 더 루프 패턴은 세 가지 핵심 구성요소를 사용합니다:
- `RequestInfoExecutor` - 워크플로를 일시 중지하고 이벤트를 발생시킴
- `RequestInfoMessage` - 타입화된 요청 페이로드의 기본 클래스 (이 클래스를 상속하세요!)
- `RequestResponse` - 인간 응답을 원래 요청과 연계시킴

**중요 이해 사항:**
- `RequestInfoExecutor`는 직접 입력을 수집하지 않습니다 - 워크플로만 일시 중지합니다
- 애플리케이션 코드는 `RequestInfoEvent`를 수신하여 입력을 수집해야 합니다
- 사용자 답변에 대해 `request_id`를 키로 하는 dict를 `send_responses_streaming()`에 전달해야 합니다

#### 2. **스트리밍 실행 패턴**
```python
# 첫 번째 반복
stream = workflow.run_stream(initial_request)

# 이후 반복(사용자 입력 후)
stream = workflow.send_responses_streaming(pending_responses)

# 항상 이벤트 처리
events = [event async for event in stream]
```

#### 3. **이벤트 기반 아키텍처**
워크플로 제어를 위해 특정 이벤트를 청취하세요:
- `RequestInfoEvent` - 인간 입력이 필요함 (워크플로 일시 중지)
- `WorkflowOutputEvent` - 최종 결과가 사용 가능함 (워크플로 완료)
- `WorkflowStatusEvent` - 상태 변경 (IN_PROGRESS, IDLE_WITH_PENDING_REQUESTS 등)

#### 4. **@handler를 이용한 맞춤 실행기**
`DecisionManager`는 다음을 보여줍니다:
- `@handler` 데코레이터로 메서드를 워크플로 단계로 노출
- 타입화된 메시지 수신 (예: `RequestResponse[HumanFeedbackRequest, str]`)
- 메시지를 다른 실행기로 보내 워크플로를 라우팅
- `WorkflowContext`를 통해 컨텍스트 접근

#### 5. **휴먼 결정에 따른 조건부 라우팅**
인간 응답을 평가하는 조건 함수 생성이 가능합니다:
```python
def user_wants_alternatives_condition(message: Any) -> bool:
    response_text = message.agent_run_response.text.lower()
    return response_text == "yes"
```

### 🎯 실제 응용 사례:

1. **승인 워크플로**
   - 지출 보고서 처리 전에 관리자 승인 받기
   - 자동 이메일 전송 전에 인간 검토 요구
   - 고액 거래 실행 전에 확인

2. **콘텐츠 검토**
   - 문제 콘텐츠를 인간 검토 위해 표시
   - 경계 케이스에 대해 검토자가 최종 결정
   - AI 신뢰도가 낮을 때 인간에게 에스컬레이션

3. **고객 서비스**
   - 일상적인 질문은 AI가 자동으로 처리
   - 복잡한 문제는 인간 상담원에게 에스컬레이션
   - 고객에게 인간 상담원 연결 여부 질문

4. **데이터 처리**
   - 불명확한 데이터 항목을 인간이 해결하도록 요청
   - 불확실한 문서 해석에 대해 AI 해석 확인
   - 사용자에게 여러 유효 해석 중 선택 권한 제공

5. **안전-critical 시스템**
   - 되돌릴 수 없는 작업 전에 인간 확인 요구
   - 민감한 데이터 접근 전에 승인 받기
   - 규제 산업(의료, 금융)에서 결정 확인

6. **인터랙티브 에이전트**
   - 후속 질문을 하는 대화형 봇 구축
   - 복잡한 프로세스를 안내하는 마법사 생성
   - 인간과 단계별로 협력하는 에이전트 설계

### 🔄 비교: 휴먼 인 더 루프 유무

| 특성 | 조건부 워크플로 | 휴먼 인 더 루프 워크플로 |
|---------|---------------------|---------------------------|
| <strong>실행</strong> | 단일 `workflow.run()` | `run_stream()` + `send_responses_streaming()`을 포함한 루프 |
| **사용자 입력** | 없음 (완전 자동화) | `input()` 또는 UI를 통한 대화형 프롬프트 |
| <strong>구성요소</strong> | 에이전트 + 실행기 | + RequestInfoExecutor + DecisionManager |
| <strong>이벤트</strong> | AgentExecutorResponse만 | RequestInfoEvent, WorkflowOutputEvent 등 |
| **일시 중지** | 없음 | RequestInfoExecutor에서 워크플로 일시 중지 |
| **인간 제어** | 없음 | 인간이 주요 결정을 내림 |
| **사용 사례** | 자동화 | 협업 및 감독 |

### 🚀 고급 패턴:

#### 다중 인간 결정 지점
동일 워크플로에 여러 `RequestInfoExecutor` 노드를 둘 수 있습니다:
```python
.add_edge(agent1, request_info_1)  # 첫 번째 인간 결정
.add_edge(decision_manager_1, agent2)
.add_edge(agent2, request_info_2)  # 두 번째 인간 결정
.add_edge(decision_manager_2, final_agent)
```

#### 타임아웃 처리
인간 응답에 대한 타임아웃을 구현하세요:
```python
import asyncio

try:
    answer = await asyncio.wait_for(
        asyncio.to_thread(input, "Enter yes/no: "),
        timeout=60.0
    )
except asyncio.TimeoutError:
    answer = "no"  # 기본값은 안전한 옵션으로 설정
```

#### 풍부한 UI 통합
`input()` 대신 웹 UI, Slack, Teams 등과 통합하세요:
```python
if isinstance(event, RequestInfoEvent):
    # 사용자가 선호하는 채널로 알림을 전송합니다
    await slack_client.send_message(
        user_id=current_user,
        text=event.data.prompt,
        request_id=event.request_id
    )
```

#### 조건부 휴먼 인 더 루프
특정 상황에서만 인간 입력 요청:
```python
def needs_human_approval_condition(message: Any) -> bool:
    # 신뢰도가 낮거나 값이 높을 때만 사람에게 경로를 지정합니다
    if result.confidence < 0.7 or result.value > 10000:
        return True
    return False
```

### ⚠️ 모범 사례:

1. **항상 RequestInfoMessage를 서브클래스화하세요**
   - 타입 안전성과 검증 제공
   - UI 렌더링을 위한 풍부한 컨텍스트 가능
   - 각 요청 유형의 의도를 명확히 함

2. **설명적인 프롬프트 사용**
   - 요청하는 내용에 대한 컨텍스트 포함
   - 각 선택의 결과 설명
   - 질문을 간단하고 명확하게 유지

3. **예상치 못한 입력 처리**
   - 사용자 응답 검증
   - 잘못된 입력에 대한 기본값 제공
   - 명확한 오류 메시지 제공

4. **요청 ID 추적**
   - request_id와 응답 간의 상관관계 사용
   - 상태를 수동으로 관리하려 하지 마세요

5. **비차단 설계**
   - 입력 대기 시 스레드 차단하지 않기
   - 전반에 걸쳐 비동기 패턴 사용
   - 동시 워크플로 인스턴스 지원

### 📚 관련 개념:

- **에이전트 미들웨어** - 에이전트 호출 가로채기 (이전 노트북)
- **워크플로 상태 관리** - 실행 간 워크플로 상태 지속
- **다중 에이전트 협력** - 휴먼 인 더 루프와 에이전트 팀 결합
- **이벤트 기반 아키텍처** - 이벤트로 반응형 시스템 구축

---

### 🎓 축하합니다!

마이크로소프트 에이전트 프레임워크를 활용한 휴먼 인 더 루프 워크플로를 마스터했습니다! 이제 다음을 할 수 있습니다:
- ✅ 인간 입력을 수집하기 위해 워크플로를 일시 중지
- ✅ RequestInfoExecutor 및 RequestInfoMessage 사용
- ✅ 이벤트를 활용한 스트리밍 실행 처리
- ✅ @handler로 맞춤 실행기 생성
- ✅ 인간 결정에 기반한 워크플로 라우팅
- ✅ 인간과 협업하는 인터랙티브 AI 에이전트 구축

**이는 신뢰할 수 있고 제어 가능한 AI 시스템 구축을 위한 핵심 패턴입니다!** 🚀


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**면책 조항**:
이 문서는 AI 번역 서비스 [Co-op Translator](https://github.com/Azure/co-op-translator)를 사용하여 번역되었습니다. 정확성을 기하기 위해 노력하고 있으나, 자동 번역은 오류나 부정확한 부분이 있을 수 있음을 유의하시기 바랍니다. 원본 문서의 원어본이 권위 있는 자료로 간주되어야 합니다. 중요한 정보의 경우, 전문가의 인간 번역을 권장합니다. 이 번역 사용으로 인해 발생하는 오해나 잘못된 해석에 대해 당사는 책임을 지지 않습니다.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
